# Task 6 — Idempotent Replay Verification

**Points**: 1.0 / 10  
**Scripts**: [`src/parser_service.py`](../../src/parser_service.py), [`src/spark_streaming_job.py`](../../src/spark_streaming_job.py)

---

## 6.1 Method

To demonstrate that reprocessing a single modified file does not create duplicates and that the streaming job resumes correctly, we:

1. Capture the **baseline** state of one file's document in MongoDB
2. **Modify** that file inside the cloned `repos/lerobot` checkout
3. **Reprocess only that file** through the Parser Service (`--file` flag — not a full repo re-run)
4. **Restart** the Spark Structured Streaming job and confirm it resumes from the last committed offset rather than reprocessing all 545 files
5. Verify the MongoDB document was **updated in place**, with no increase in total document count

Target file: `lerobot/examples/annotations/run_hf_job.py` (`rel_path` in `discovered_files.json`: `examples/annotations/run_hf_job.py`).

## 6.2 Baseline (Before Replay)

In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client.cpg_db
print("Total docs (before):", db.cpg_metadata.count_documents({}))
print("Before doc:", db.cpg_metadata.find_one({"_id": "lerobot/examples/annotations/run_hf_job.py"}))

Total docs (before): 545
Before doc: {'_id': 'lerobot/examples/annotations/run_hf_job.py', 'schema_version': '1.0', 'event_time': '2026-07-22T08:59:02.474030+00:00', 'event_type': 'metadata', 'file_hash': 'f5aad3e886a796e3e885b36ba8f511ebdd1540058cdf61073e0526e4a0b3ccb0', 'size_bytes': 3158, 'loc': 81, 'num_functions': 0, 'num_classes': 0, 'num_nodes': 62, 'num_edges': 21}


## 6.3 Modifying the Source File

A single line is appended to the real file inside the cloned repository — enough to change its content hash and line count without altering its AST structure meaningfully:

# echo '# task6 replay test comment' >> repos/lerobot/examples/annotations/run_hf_job.py
# tail -3 repos/lerobot/examples/annotations/run_hf_job.py

...
# task6 replay test comment


## 6.4 Reprocessing Only the Modified File

`parser_service.py --file <rel_path>` processes exactly one file rather than the full 545-file repository — this is the mechanism that makes the CPG parser side of replay incremental:

# python src/parser_service.py --file examples/annotations/run_hf_job.py

[1/1] processed (0.1s elapsed)
Done. 1 files parsed OK, 0 errors, 62 nodes, 21 edges emitted.


This emits exactly one new message onto `cpg.metadata` (offset 545, since offsets 0–544 were already committed from the Task 5 run), with the updated `file_hash` and `loc`.

## 6.5 Resuming the Spark Streaming Job — Checkpoint Proof

Re-running the exact same command as Task 5 (`python src/spark_streaming_job.py`, `availableNow` mode) with the **same checkpoint directory**:

Streaming query finished. Final progress: {
  'id': 'dabaa5ba-2403-4d20-8316-bf1471cc3ea8',
  'runId': 'a70d605d-c3c0-48af-b08c-760cb28ad6e5',
  'batchId': 1,
  'numInputRows': 1,
  'sources': [{
      'description': 'KafkaV2[Subscribe[cpg.metadata]]',
      'startOffset': {'cpg.metadata': {'0': 545}},
      'endOffset': {'cpg.metadata': {'0': 546}},
      'numInputRows': 1
  }],
  'sink': {
      'description': 'MongoTable{... spark.mongodb.collection=cpg_metadata ...}',
      'numOutputRows': 1
  }
}


This is the key evidence for checkpoint-based idempotent replay:

| Field | Task 5 (first run) | Task 6 (replay) |
|---|---|---|
| `batchId` | 0 | **1** (continues the sequence, not restarted) |
| `startOffset` | `None` | **`{"0": 545}`** (resumes exactly where it left off) |
| `endOffset` | `{"0": 545}` | `{"0": 546}` |
| `numInputRows` | 545 | **1** (only the newly produced message, not all 546) |
| `numOutputRows` | 545 | **1** |

The job never re-read the 545 unchanged files' offsets — confirming the checkpoint correctly skips already-processed offsets, exactly as Task 6 requires.

## 6.6 Verifying the Updated Document (No Duplication)

In [5]:
print("Total docs (after):", db.cpg_metadata.count_documents({}))
print("After doc:", db.cpg_metadata.find_one({"_id": "lerobot/examples/annotations/run_hf_job.py"}))

Total docs (after): 545
After doc: {'_id': 'lerobot/examples/annotations/run_hf_job.py', 'schema_version': '1.0', 'event_time': '2026-07-22T09:06:09.750528+00:00', 'event_type': 'metadata', 'file_hash': '4a955f9a3afdce774eec2d54bd1b335bdc86ec1065a54a78d76795b7e43930b1', 'size_bytes': 3186, 'loc': 82, 'num_functions': 0, 'num_classes': 0, 'num_nodes': 62, 'num_edges': 21}


Three facts confirm idempotent replay succeeded:

1. **`Total docs` unchanged: 545 → 545.** No new document was inserted for the replayed file.
2. **`file_hash` changed**: `f5aad3e8...` → `4a955f9a...` — the document content genuinely reflects the file's new state, it wasn't just left stale.
3. **`loc` incremented**: `81` → `82`, matching the one appended comment line exactly.

*(Insert a MongoDB Compass screenshot showing this single document's history/updatedAt here, and a Neo4j Browser screenshot confirming node/edge counts for this file are unchanged, for the final submission.)*

## 6.7 Checkpoint Files on Disk

As additional evidence that the checkpoint is a real, persisted mechanism and not just an in-memory artifact of a single process:

# ls checkpoints/cpg-metadata/offsets/
0  1


Two offset log files exist — `0` for the initial 545-message batch and `1` for the single-message replay batch — confirming Spark Structured Streaming's write-ahead offset log is what allows the job to be killed and restarted at any point without reprocessing already-committed data.

## 6.8 Reflection

### What worked
- Using `_id = file_path` in MongoDB and a single-file `--file` mode in the Parser Service meant Task 6 required **no special-case code** — the same Task 5 pipeline handled replay correctly by design.
- The Spark checkpoint's `batchId`/`startOffset` progression gave a clean, quantitative way to prove "no reprocessing of unchanged files" instead of just asserting it.

### Issues encountered
- Needed to double check the exact `rel_path` format expected by `--file` (relative to `repos/lerobot`, without a `lerobot/` prefix) by cross-referencing `data/discovered_files.json`, since the Kafka message's `file_path` field uses a different, prefixed form (`lerobot/examples/...`).

### Lessons learned
- Idempotent replay is really a property of the **whole pipeline's key design** (natural key at the parser, Kafka partitioning by that key, upsert-by-`_id` at the sink) rather than something bolted on afterward — verifying it end-to-end is what this task is really testing.